<!-- NB_DOC_INTRO_V1 -->
# NLP — Named Entity Recognition (NER)

> 📚 **Doc thematique** : [docs/05_NLP.md](docs/05_NLP.md) (NLP)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**NER** = identifier les entites nommees dans un texte (Personne, Lieu, Organisation, Date, ...). Couvre les **3 approches** : (1) **regles** + dictionnaires (baseline), (2) **classique ML** avec features (CRF, sklearn), (3) **deep learning** (BiLSTM-CRF, Transformers pre-entraine).

Ce notebook execute spaCy (le standard prod) sur du texte FR + montre les formats d'annotation (IOB / BIOES).

Pour BiLSTM-CRF from-scratch : [NLP_NER_BiLSTM_CRF.ipynb](./NLP_NER_BiLSTM_CRF.ipynb).

## Plan

1. Concepts (entites, IOB / BIOES tagging schemes)
2. Demo spaCy (FR + EN, pre-entraine)
3. Custom NER avec spaCy (entrainement)
4. Eval avec seqeval
5. Format IOB / BIOES
6. HuggingFace pre-trained NER (transformers)
7. Top modeles FR / EN 2024-2025
8. Pieges + References


In [ ]:
import numpy as np
import warnings
warnings.filterwarnings("ignore")
SEED = 42

## 1. Concepts NER + tagging schemes

### Entites standard (CoNLL-2003)
| Tag | Description |
|---|---|
| `PER` | Personne |
| `LOC` | Lieu |
| `ORG` | Organisation |
| `MISC` | Divers (nationalites, events, ...) |
| `DATE` | Date (spaCy en + d'autres) |
| `MONEY` | Montant |
| `O` | Outside (token sans entite) |

### Schemes de tagging par token

| Scheme | Exemple "Barack Obama est ne aux USA" |
|---|---|
| **IOB1** | `Barack=I-PER  Obama=I-PER  est=O  ne=O  aux=O  USA=I-LOC` |
| **IOB2** (= BIO) | `Barack=B-PER  Obama=I-PER  est=O  ne=O  aux=O  USA=B-LOC` (B au debut de chaque entite) |
| **BIOES** (= BILOU) | `Barack=B-PER  Obama=E-PER  est=O  ne=O  aux=O  USA=S-LOC` (E=End, S=Single) |

BIOES est **plus expressif** que BIO (distingue "fin d'entite" du "milieu"), souvent meilleurs scores.

## 2. Demo spaCy — modele pre-entraine

In [ ]:
# spaCy : le standard prod, leger, multilingue
try:
    import spacy
    SPACY_OK = True
    print(f"spaCy : {spacy.__version__}")
except ImportError:
    SPACY_OK = False
    print("spaCy pas installe : uv add --group nlp spacy")

# Le modele FR/EN doit etre telecharge separement : python -m spacy download fr_core_news_md
# Pour la demo, on teste si dispo
SPACY_FR_OK = False
if SPACY_OK:
    try:
        nlp_fr = spacy.load("fr_core_news_sm")
        SPACY_FR_OK = True
        print("Modele FR small charge.")
    except OSError:
        print("Modele fr_core_news_sm non trouve.")
        print("Installer : python -m spacy download fr_core_news_sm")

In [ ]:
# === Demo NER sur du texte FR (si modele dispo) ===
texts_fr = [
    "Emmanuel Macron a rencontre Joe Biden a Washington le 15 janvier 2024.",
    "Apple a annonce un nouveau iPhone 15 a 1200 euros lors d'une conference a Cupertino.",
]

if SPACY_FR_OK:
    for text in texts_fr:
        doc = nlp_fr(text)
        print(f"Texte : {text}")
        for ent in doc.ents:
            print(f"  → {ent.text:30s} | {ent.label_:8s} | start={ent.start_char}, end={ent.end_char}")
        print()
else:
    print("Demo skip (modele FR non dispo).")
    print("Output attendu :")
    print("  Emmanuel Macron  | PER")
    print("  Joe Biden        | PER")
    print("  Washington       | LOC")
    print("  15 janvier 2024  | DATE")

## 3. Eval NER avec `seqeval` (la lib reference)

In [ ]:
try:
    from seqeval.metrics import classification_report as seq_report, f1_score as seq_f1
    from seqeval.metrics import precision_score, recall_score

    # Format : liste de liste de tags par phrase (IOB2)
    y_true = [
        ["B-PER", "I-PER", "O", "B-LOC", "O"],
        ["B-ORG", "O", "B-PER", "I-PER", "O"],
    ]
    y_pred = [
        ["B-PER", "I-PER", "O", "B-LOC", "O"],
        ["B-ORG", "O", "B-PER", "O", "O"],         # 1 erreur sur la 2e phrase
    ]

    print(seq_report(y_true, y_pred, digits=3))
    print(f"F1 (entity-level): {seq_f1(y_true, y_pred):.4f}")
except ImportError:
    print("seqeval pas installe : uv add --group nlp seqeval")

**IMPORTANT** : utiliser **seqeval** (qui evalue **par entite complete**), pas `sklearn.metrics.f1_score` (qui evalue token par token et **surestime** la perf).

## 4. Custom NER avec spaCy (entrainement)

```python
import spacy
from spacy.training import Example

# Donnees d'entrainement : (text, {"entities": [(start, end, label), ...]})
TRAIN_DATA = [
    ("Apple vend des iPhones a Paris", {"entities": [(0, 5, "ORG"), (25, 30, "LOC")]}),
    ("Google a son siege a Mountain View", {"entities": [(0, 6, "ORG"), (22, 35, "LOC")]}),
    # ... ~ 100-500 exemples typique
]

# Blank model
nlp = spacy.blank("en")
ner = nlp.add_pipe("ner")
ner.add_label("ORG"); ner.add_label("LOC")

# Train
optimizer = nlp.begin_training()
for itn in range(30):
    losses = {}
    for text, annotations in TRAIN_DATA:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annotations)
        nlp.update([example], drop=0.5, sgd=optimizer, losses=losses)
    print(f"Iter {itn} losses : {losses}")

# Save
nlp.to_disk("./my_ner_model")
```


## 5. HuggingFace transformers — NER pre-entraine SOTA

Pour des resultats **vraiment top** : utiliser un modele Transformers fine-tune sur NER (BERT, RoBERTa, CamemBERT...).

```python
from transformers import pipeline

# Pre-trained model (auto-DL)
ner = pipeline("ner", model="dbmdz/bert-large-cased-finetuned-conll03-english",
               aggregation_strategy="simple")
result = ner("My name is Sarah and I live in London")
# [{'entity_group': 'PER', 'word': 'Sarah', 'score': 0.99, ...},
#  {'entity_group': 'LOC', 'word': 'London', 'score': 0.99, ...}]

# Pour le francais : "Jean-Baptiste/camembert-ner"
ner_fr = pipeline("ner", model="Jean-Baptiste/camembert-ner", aggregation_strategy="simple")
ner_fr("Emmanuel Macron est ne a Amiens.")
```

### Top modeles 2024-2025

| Modele | Langue | Score CoNLL F1 |
|---|---|---|
| `dbmdz/bert-large-cased-finetuned-conll03-english` | EN | ~ 92 |
| `Jean-Baptiste/camembert-ner` | FR | ~ 90 |
| `Davlan/xlm-roberta-base-ner-hrl` | Multi (10 langues) | ~ 88 |
| `flair/ner-english-large` | EN | ~ 93 |
| `tner/twitter-roberta-base-dec2021-tweetner7` | EN (Twitter specifique) | ~ 67 |

## 6. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| Eval avec `sklearn.metrics.f1_score` (token-level) | Toujours **seqeval** (entity-level) |
| Confondre IOB1 et IOB2 | Ne pas melanger les schemes |
| Pas de `aggregation_strategy="simple"` dans pipeline HF | Output token-by-token sans regrouper |
| Tagging incomplet (oublier des entites) | NER suppose annotation **exhaustive** |
| Tester sur la meme langue qu'entrainement uniquement | Si multi-langue : modele multi-langue (XLM-R) |
| Entrainer custom NER avec < 100 exemples | Insuffisant — utiliser pre-trained et adapter |
| Pas de **post-traitement** | Normalisation des entites (capitalisation, accents) souvent utile |

## 7. References

- spaCy : https://spacy.io/
- HuggingFace NER tutorial : https://huggingface.co/learn/nlp-course/chapter7/2
- seqeval : https://github.com/chakki-works/seqeval
- CoNLL-2003 dataset : https://www.clips.uantwerpen.be/conll2003/ner/
- Flair NLP : https://github.com/flairNLP/flair

### Voir aussi (collection)
- [NLP_NER_BiLSTM_CRF.ipynb](NLP_NER_BiLSTM_CRF.ipynb) — archi BiLSTM-CRF detaillee
- [NLP_Transformers.ipynb](NLP_Transformers.ipynb) — base transformers
- [AI_LLM_Finetuning_PEFT_LoRA.ipynb](AI_LLM_Finetuning_PEFT_LoRA.ipynb) — fine-tuner pour NER custom
